In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import cartopy
import cartopy.crs as ccrs
import matplotlib.path as mpath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.feature as cfeature

import xarray as xr
from glob import glob
import pandas as pd
import numpy as np

from scipy import interpolate
from sklearn.linear_model import LinearRegression
import statsmodels.api as stats_m

import datetime as dt
import warnings
import pyproj
from scipy import signal
import importlib
import gsw as gsw

In [ ]:
from scipy.io import loadmat
import h5py
import rioxarray as rxr

In [ ]:

import Outlier_detection_v3 as Outlier_v2
importlib.reload(Outlier_v2)

In [ ]:
import os
import scipy.stats as stats 

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon, Point
import seaborn as sns

In [ ]:
from pylr2 import regress2


## functions for isopycnal regridding 

In [ ]:
## calculate vertical diffusive flux from one profile to the next
def Prep_data_may25update(ds, P_slice):
   
    print(ds.Oxygen.size)
    #print(ds.Oxygen.dropna(dim="N_LEVELS").count())
    # Check if over 85% of the DataArray is np.nan
    nan_percentage = np.sum(np.isnan(ds.Oxygen.values)) / ds.Oxygen.size
    threshold_percentage = 0.85
    if nan_percentage > threshold_percentage or len(ds.Oxygen.JULD.values)<3:
        print(f"Over {threshold_percentage * 100}% of the DataArray is np.nan.")
        print(ds.Float_ID.values)
        ds_Pgrid=xr.Dataset()
        return ds_Pgrid
    else:
        ds=calc_mld(ds, 10,.03)
        #Depth difference between levels in m
        Z_diff=ds.Depth.diff(dim="N_LEVELS")
        centered_Depth=ds.Depth.isel(N_LEVELS=slice(1, None))-Z_diff/2
        centered_Depth_dz=centered_Depth.diff(dim="N_LEVELS")
        Oz_diff=ds.Oxygen.diff(dim="N_LEVELS", n=2)
        #Timestep in seconds
        t_step_sec=ds.JULD.diff(dim="JULD")/ (np.timedelta64(1, 's'))
        #print(t_step_sec)
        F_diff=(10**-5*(Oz_diff/centered_Depth_dz).isel(JULD=slice(1, None))*(t_step_sec))
        #print(F_diff.max(), "FDIFF MAX")
        #trim upper level and first profile of the dataset so same shape as F_diff
        trimmed_DS=ds.isel(JULD=slice(1,None), N_LEVELS=slice(1,-1))
        #print(trimmed_DS.Oxygen)
        #assign Fdiff
        trimmed_DS["F_diff"]=xr.DataArray(F_diff, coords={"JULD":F_diff.JULD})

        #1-dbar pressure interp
        ds_Pgrid=dsvars_togrid_v2(trimmed_DS ,1, Pressure=True)
        #F-diff is subtracted from the profiling following the one which it is calculated from, 
        #ie, subtracting the change in concetration between profiles due to vert mixing
        new_bioOx=(ds_Pgrid.Oxygen.isel(JULD=slice(1,None)).values-ds_Pgrid.F_diff.isel(JULD=slice(0, -1)).values)
        new_bioOx_da=xr.DataArray(new_bioOx, coords=ds_Pgrid.Oxygen.isel(JULD=slice(1,None)).coords)
        ds_Pgrid["bioOxygen"]=new_bioOx_da
        ds_Pgrid=ds_Pgrid.assign_coords({"Pressure":ds_Pgrid.Pressure})
        ds_Pgrid=ds_Pgrid.sel(Pressure=P_slice)

        return ds_Pgrid


In [ ]:
def O2Fluxes_v6(ds, k, depthbelowMLD, nitrate=False):
    """
    Calculate oxygen fluxes for density layers below the mixed layer depth (MLD).
    Assumes dataset is interpolated to standard density levels.
    """
    # Time steps in seconds
    timestep = (ds.JULD.diff(dim="JULD") / np.timedelta64(1, 's')).values
    # len using every other Sigma_theta level (odd indices)
    n_layers = len(range(1,len(ds.Sigma_theta.values)-2,2))*2 #skipped layers will be zeros, those layers are removed before saveout
    n_times = len(ds.JULD)
    # Preallocate zero arrays
    LayerSigma = np.zeros(n_layers) 
    LayerStart = np.zeros(n_layers) 
    LayerStop = np.zeros(n_layers) 
    Layer_S = np.zeros((n_times, n_layers))
    Ox_in = np.zeros((n_times, n_layers))
    bioOx_in = np.zeros((n_times, n_layers))
    F_diff_P = np.zeros((n_times, n_layers))
    LayerDepth = np.zeros((n_times, n_layers))
    LayerThickness = np.zeros((n_times, n_layers))
    #depth_inv_layer= np.zeros((n_times, n_layers)) # use if you want to keep track of density inversions 

    #Loop one=find start and stop (note, each layer needs a upper and lower to calculate thickness)
    for layer in range(1,len(ds.Sigma_theta.values)-2,2):
        #print(ds.Sigma_theta[layer].values, "Layer ST")
        LayerSigma[layer]=ds.Sigma_theta[layer].values
        ds_subMLD_Oxygen=ds.Oxygen.isel(Sigma_theta=layer).where(ds.isel(Sigma_theta=layer).Depth>ds.MLD.interpolate_na(dim="JULD")+depthbelowMLD, drop=False)
        if ds_subMLD_Oxygen.isnull().all()==True:
            LayerStart[layer]="Nan"
            LayerStop[layer]="Nan"
            continue
        else:
            #find when layer is below MLD
            a = ds_subMLD_Oxygen.values
            amask= np.concatenate(( [True], np.isnan(a), [True] ))  # concatenated mask that has True values at group boundaries (groups decided by the occurrences of NaNs). That's needed in the next step to get Start-stop limits
            ss = np.flatnonzero(amask[1:] != amask[:-1]).reshape(-1,2)   # Start-stop limits
            LayerStart[layer],LayerStop[layer] = ss[(ss[:,1] - ss[:,0]).argmax()]  # Get max interval, interval limits

    #Loop 2-calc check for density inversions, replace with np.nan if found, otherwise convert oxygen to mol m3
    for t in range(len(ds.JULD.values)):
        #These are only done between start and stop
        for layer in range(1,len(ds.Sigma_theta.values)-2,2):
            if t>=LayerStart[layer] and t<LayerStop[layer]:
                if ds.Depth.isel(JULD=t, Sigma_theta=int(layer)+1)<ds.Depth.isel(JULD=t, Sigma_theta=int(layer)-1):
                    print(LayerDepth[t, layer], "warning, layer depth inversion")
                    LayerDepth[t, layer]=np.nan
                    LayerThickness[t, layer]=np.nan
                    Ox_in[t,layer]=np.nan
                    bioOx_in[t,layer]=np.nan
                    F_diff_P[t,layer]=np.nan
                    Layer_S[t, layer]=np.nan
                    #depth_inv_layer[t, layer]=1
                else:    
                    LayerDepth[t, layer]=ds.Depth.isel(JULD=t, Sigma_theta=int(layer)).values
                    LayerThickness[t, layer]=ds.Depth.isel(JULD=t, Sigma_theta=int(layer)+1)-ds.Depth.isel(JULD=t, Sigma_theta=int(layer)-1)
                    Ox_in[t,layer]=ds.Oxygen.isel(JULD=t).isel(Sigma_theta=int(layer))/1000000*(ds["Sigma_theta"].isel(Sigma_theta=int(layer))+1000) #Oxygen mol/m3 of layer
                    bioOx_in[t,layer]=ds.bioOxygen.isel(JULD=t).isel(Sigma_theta=int(layer))/1000000*(ds["Sigma_theta"].isel(Sigma_theta=int(layer))+1000) #bioOxygen mol/m3 of layer
                    F_diff_P[t,layer]=ds.F_diff.isel(JULD=t).isel(Sigma_theta=int(layer))/1000000*(ds["Sigma_theta"].isel(Sigma_theta=int(layer))+1000) #diffusive flux mol/m3 of layer
                    Layer_S[t, layer]=ds.Salinity.isel(JULD=t).isel(Sigma_theta=int(layer))
                    #depth_inv_layer[t, layer]=0
            else:
                LayerDepth[t, layer]=0
                LayerThickness[t,layer]=0
                Ox_in[t,layer]=0
                bioOx_in[t,layer]=0
                F_diff_P[t,layer]=0
                Layer_S[t, layer]=0
                #depth_inv_layer[t, layer]=0
                
    #depth_inv_layer=xr.DataArray(depth_inv_layer, dims=["JULD", "Layer"])
    #print(depth_inv_layer.sum(dim="Layer"), "sum layer")
    
    # Build output dataset
    ds_out=xr.Dataset( coords=dict(JULD=(["JULD"], ds.JULD.values), Sigma_theta=(["Layer"],LayerSigma)))
    ds_out["LayerThickness"]=xr.DataArray(LayerThickness, dims=["JULD", "Layer"])
    ds_out["LayerDepth"]=xr.DataArray(LayerDepth, dims=["JULD", "Layer"])
    ds_out["Oxygen"]=xr.DataArray(Ox_in, dims=["JULD", "Layer"])
    ds_out["bioOxygen_pressure"]=xr.DataArray(bioOx_in, dims=["JULD", "Layer"])
    ds_out["Fdiff_pressure"]=xr.DataArray(F_diff_P, dims=["JULD", "Layer"])
    ds_out["LayerSal"]=xr.DataArray(Layer_S, dims=["JULD", "Layer"])
    ds_out["LayerSigma"]=xr.DataArray(LayerSigma, dims=["Layer"])
    ds_out["MLD"]=xr.DataArray(ds.MLD.values, dims=["JULD"])
    ds_out["Lat"]=xr.DataArray(ds.Lat.values, dims=["JULD"])
    ds_out["Lon"]=xr.DataArray(ds.Lon.values, dims=["JULD"])
    ds_out=ds_out.swap_dims({"Layer":"Sigma_theta"})
    ds_out=ds_out.where(ds_out.LayerDepth>0, drop=True)
    ds_out["seasonID"]=str(ds.Float_ID.values)[0:7]+"_"+str(ds.JULD[0].dt.year.values)
    ds_out=ds_out.assign_coords({"seasonID":str(ds.Float_ID.values)[0:7]+"_"+str(ds.JULD[0].dt.year.values)})
    
    return ds_out


In [ ]:
def LOESS_filter(ds, param):
    days = ds.JULD.values
    df_LOESS = pd.DataFrame(columns=days, index=np.arange(len(ds.Sigma_theta.values)))
    for k in range(len(ds.Sigma_theta.values)):
        layer=ds.isel(Sigma_theta=k).Sigma_theta.values
        layer_time=np.arange(0,len(ds.sel(Sigma_theta=layer)[param].dropna(dim="JULD").values))
        layer_param=ds.sel(Sigma_theta=layer)[param].dropna(dim="JULD").values
        df_LOESS.loc[k] =pd.Series(data=stats_m.nonparametric.lowess(layer_param, layer_time, frac=0.5, it=3, delta=0.0)[:,1], index=ds.isel(Sigma_theta=k)[param].dropna(dim="JULD").JULD.values)
    return df_LOESS

## for bNCP

In [ ]:
### calc bNCPML
def bNCP_ML(ds, k=10**-4 ):
    """read in processed ds (QC'd for only good data, passed all checks)""" 
    ds=mldmean_anyspacing(ds, Snorm=True, bioOpt=True)
    if ds.ML_NSnorm.min()<=1.75:
        print("low nitrate")
        return (np.nan ,np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)
    ds_fine=ds.interp(Depth=np.arange(0,700,.1))
    ds_daily=ds_fine.resample(JULD="1D").interpolate()
    ds=ds_daily.interpolate_na(dim="JULD", method="linear", limit=2)
    t_end=ds.JULD[-1].dt.year.values
    t_start=ds.JULD[0].dt.year.values 
    if t_end==t_start:
        print(t_end, t_start)
        print("t_end == t_start")
        return (np.nan ,np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)
    elif ds.ML_NSnorm.sel(JULD=str(t_end)).count()<1:
        return (np.nan ,np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)
    else:
        MLD_stop=ds.ML_NSnorm.sel(JULD=str(t_end)).where(ds.ML_NSnorm.sel(JULD=str(t_end))==ds.ML_NSnorm.sel(JULD=str(t_end)).min().values, drop=True)[0].JULD.values
        MLD_start=ds.ML_NSnorm.sel(JULD=str(t_start)).where(ds.ML_NSnorm.sel(JULD=str(t_start))==ds.ML_NSnorm.sel(JULD=str(t_start)).max().values, drop=True)[0].JULD.values
        print(MLD_start, "MLD_start \n")
        ds=ds.sel(JULD=slice(MLD_start, ds.JULD[-1].values))
        
        Ncheck=np.zeros(len(ds.JULD))
        J=np.zeros(len(ds.JULD))
        J2=np.zeros(len(ds.JULD))
        FE=np.zeros(len(ds.JULD))
        FK=np.zeros(len(ds.JULD))
        dNdt=np.zeros(len(ds.JULD))
        dNdtsubML=np.zeros(len(ds.JULD))
        Sig=np.zeros(len(ds.JULD))

        mldi=ds.MLD
        N=ds.ML_NSnorm
        Sig=ds.ML_Sig_theta
        for t in range(1,len(ds.JULD)-2):
            NT=N[t]*(Sig[t]+1000)*mldi[t]
            NTo=N[t-1]*(Sig[t-1]+1000)*mldi[t-1]
            dNdt[t]=NT-NTo
            #print(dNdt[t], "umol N / m2 /d")
            if mldi[t]-mldi[t-1] >= 0: 
                E=mldi[t]-mldi[t-1]
                #print(E, "E deepening")
                N_b=ds.Snorm_Nitrate.isel(JULD=t-1).sel(Depth=mldi[t].values, method="nearest")
                Sig_b=ds.Sigma_theta.isel(JULD=t-1).sel(Depth=mldi[t].values, method="nearest")
                FE[t]=N_b*(Sig_b+1000)*E# #entrainment flux, [umol/kg*kg/m3*m/timestep]
                
            else: 
                E=mldi[t]-mldi[t-1]
                #print(E, "E shoaling")
                FE[t]=N[t-1]*(Sig[t-1]+1000)*E
            #print(FE[t], "FE at time t")
            #diffusive flux (eddy diffusion coefficent 10e-4 m2) * gradient from +/- 4dbar of mixed layer
            N_below=ds.Snorm_Nitrate.isel(JULD=t).sel(Depth=(mldi[t]+2), method="nearest")*(ds.Sigma_theta.isel(JULD=t).sel(Depth=(mldi[t]+2), method="nearest")+1000)
            N_above=ds.Snorm_Nitrate.isel(JULD=t).sel(Depth=(mldi[t]-2), method="nearest")*(ds.Sigma_theta.isel(JULD=t).sel(Depth=(mldi[t]-2), method="nearest")+1000)
            P_below=ds.Snorm_Nitrate.isel(JULD=t).sel(Depth=(mldi[t]+2), method="nearest").Depth.values
            P_above=ds.Snorm_Nitrate.isel(JULD=t).sel(Depth=(mldi[t]-2), method="nearest").Depth.values
            FK[t]=k*((N_above-N_below)/-4)*(86400) #m2/s * mol/m4 *day/s             

            #Residual
            J[t]=dNdt[t]-FE[t]-FK[t]-Fek[t]
            #check 
            Ncheck[t]=dNdt[t]+(N[t-1]*(Sig[t-1]+1000)*mldi[t-1]-(N[t]*(Sig[t]+1000)*mldi[t]))
        # np array to xarray for plotting
        JNO3= xr.DataArray(J,coords={"JULD":ds.JULD}, dims=["JULD"]) #umol / m2/d
        FENO3= xr.DataArray(FE,coords={"JULD":ds.JULD}, dims=["JULD"])
        FKNO3= xr.DataArray(FK,coords={"JULD":ds.JULD}, dims=["JULD"])
        dNO3dt= xr.DataArray(dNdt,coords={"JULD":ds.JULD}, dims=["JULD"])
        NO3check=xr.DataArray(Ncheck, coords={"JULD":ds.JULD}, dims=["JULD"])
        #get cumsum max
        bNCP_max=(JNO3*-1).where((JNO3*-1).cumsum()==(JNO3*-1).cumsum().max(), drop=True).JULD[0].values
        print(bNCP_max, "bNCP_max")
        JNO3_b=JNO3.sel(JULD=slice(JNO3.JULD[0].values, bNCP_max))
        bNCP_ML=(JNO3_b.sum()*(-106/16)*(10**-6)).values
        
        return bNCP_ML

### Secondary check to filter out float years with large salinity, or depth changes 
Lat/ Lon and too little data criteria are best applied before density interpolation.
Ideally, files are visually inspected as well!

In [ ]:

def S_Z_check_v3(ds_out, s_crit=0.15, z_crit=200, s_percent=.25, z_percent=.17):
    sal_condition_met = False
    depth_condition_met = False
        
    delta_S=ds_out.LayerSal.max(dim="JULD")-ds_out.LayerSal.min(dim="JULD")
    delta_z=ds_out.LayerDepth.max(dim="JULD")-ds_out.LayerDepth.min(dim="JULD")
    sal_condition_count=0
    z_condition_count=0
    for layer in range(len(ds_out.Sigma_theta)):
        if np.max(np.abs(delta_S.isel(Sigma_theta=layer).values)) >= s_crit:
            sal_condition_met = True
            sal_condition_count=sal_condition_count+1
            print(sal_condition_count, print("updated sal count"))

        elif np.max(np.abs(delta_z.isel(Sigma_theta=layer).values)) >= z_crit:
            depth_condition_met = True
            z_condition_count=z_condition_count+1
        else:
            print("no large S, depth, changes")
    if sal_condition_count/len(ds_out.Sigma_theta)>s_percent:
        scnd_sal_condition_met=True
        print(sal_condition_count/len(ds_out.Sigma_theta), "% S thresh exceeded")
        print(sal_condition_count, "S layer count, for 2nd condition met")
        print(ds_out.seasonID, "fltID")
    else: 
        scnd_sal_condition_met=False
        print(ds_out.seasonID, "2nd condition pass")
    if z_condition_count/len(ds_out.Sigma_theta)>z_percent:
        scnd_z_condition_met=True
        print(z_condition_count/len(ds_out.Sigma_theta), "% z thresh exceeded")
        print(z_condition_count, "z layer count, for 2nd condition met")
        print(ds_out.seasonID, "fltID")
    else: 
        scnd_z_condition_met=False
        print(ds_out.seasonID, "2nd z condition pass")
    return scnd_sal_condition_met, scnd_z_condition_met
        

In [ ]:
#random useful 
def convert_360_lon_to_180(lons):
    """ Converts any-dimension array of longitudes from 0 to 360 to longitudes from -180 to 180.
    """
    lons = np.array(lons)
    outside_range = lons > 180
    lons[outside_range] = lons[outside_range] - 360
    return lons

def convert_180_lon_to_360(lons):
    """ Converts any-dimension array of longitudes from -180 to 180 t0 0 to 360 to longitude.
    """
    lons = np.array(lons)
    outside_range = lons < 0
    lons[outside_range] = lons[outside_range] + 360
    return lons


### Main functions to find pot. density layers below seasonal ML and estimate linear regression between max and min 

In [ ]:
def R_LinearRegression(time, oxygen, level_ST, saveout=False):
    X = time.reshape((-1, 1))
    y = oxygen
    model = LinearRegression()
    model.fit(X, y)
    yfit = model.predict(X)
    
    r_sq=model.score(X, y)
    return model.coef_, r_sq, model.intercept_

def R_and_NCP_v3(ds, smooth_bioO2=False, saveplots=False):
    sig_theta_dim_len=len(ds.Sigma_theta.values)
    RLinear_slope = np.zeros(sig_theta_dim_len)
    R_intercept = np.zeros(sig_theta_dim_len)
    OnesLike = np.zeros([len(ds.JULD.values), sig_theta_dim_len])
    R_R2 = np.zeros(sig_theta_dim_len)
    layer_ST = np.zeros(sig_theta_dim_len)
    R_ProfNum = np.zeros(sig_theta_dim_len)
    R_tstep_days = np.zeros(sig_theta_dim_len)
    outliers_perlayer=np.zeros([sig_theta_dim_len, 100])
    
    for layer in range(len(ds.Sigma_theta.values)):
        a = ds.bioOxygen_pressure.isel(Sigma_theta=layer).to_series()
        # Remove Outliers
        max2, tested, outliers_perlayer_sequence = Outlier_v2.ESD_Test(a.values, 0.05, (a.count() * 0.05).astype(int), True, True)
        arr_app=np.append(outliers_perlayer_sequence, (np.repeat(0, 100-len(outliers_perlayer_sequence))))
        outliers_perlayer[layer,:]=arr_app
        
        if max2 > 0:
            # Replace outliers with np.nan
            for k in range(len(outliers_perlayer_sequence)):
                #ds['bioOxygen_filtered'] = ds['bioOxygen'].where(ds['bioOxygen'] != outliers_perlayer[layer, k], np.nan)
                ds['bioOxygen_filtered'] = ds['bioOxygen_pressure'].where(ds['bioOxygen_pressure'] != outliers_perlayer[layer, k], np.nan)
    if "bioOxygen_filtered" in ds.variables.keys():
        print("outliers present")
    else:
        ds["bioOxygen_filtered"]=ds.bioOxygen_pressure.copy()
        print("checked for outliers but none found")
    
    # smoothing
    if smooth_bioO2 == True:
        LOESSsmoothed_pd = LOESS_filter(ds, "bioOxygen_filtered").T
        bioO2_smoothed = xr.DataArray(data=LOESSsmoothed_pd.values.astype(float), dims=["JULD", "Sigma_theta"], coords={"JULD": LOESSsmoothed_pd.index, "Sigma_theta": ds.Sigma_theta.values})
        ds["bioOxygen_4ncp"] = bioO2_smoothed
    else:
        ds["bioOxygen_4ncp"] = ds["bioOxygen_filtered"].copy()
    
    for layer in range(len(ds.Sigma_theta.values)):               
        #find max to min 
        layer_maxT=ds.isel(Sigma_theta=layer).where(ds.isel(Sigma_theta=layer).bioOxygen_4ncp==ds.isel(Sigma_theta=layer).bioOxygen_4ncp.max(),drop=True).JULD[0]
        aftermax=ds.isel(Sigma_theta=layer).sel(JULD=slice(layer_maxT.values, ds.JULD[-1].values))
        layer_minaftermaxT=aftermax.where(aftermax.bioOxygen_4ncp==aftermax.bioOxygen_4ncp.min(),drop=True).JULD[0]
        layerconsumption=aftermax.sel(JULD=slice(layer_maxT.values, layer_minaftermaxT.values))
        
        #find number of timesteps and check evenly spaced
        if layerconsumption.JULD.diff(dim="JULD").dt.days.std().values>1:
            print("WARNING UNEVEN TSTEPS!")
            print(layerconsumption.JULD.values)
            #uneven_t_sample_flt.append(str(ds.seasonID.values))
            layer_time=np.arange(0,len(layerconsumption.bioOxygen_4ncp.resample(JULD="10D").interpolate()))
            print(layerconsumption.bioOxygen_4ncp.resample(JULD="10D").interpolate().JULD, "new time")
            print(layer_time, "new time step")
            layerconsumption=layerconsumption.resample(JULD="10D").interpolate()
        else:
            layer_time=np.arange(0,len(layerconsumption.JULD.values))
        if len(layerconsumption.JULD.values)<=2:
            print("only one timestep")
            RLinear_slope[layer], R_R2[layer],R_intercept[layer]=np.nan, np.nan, np.nan
            layer_ST[layer]=ds.isel(Sigma_theta=layer).Sigma_theta.values
            OnesLike[:, layer]=np.repeat(np.nan, len(ds.JULD.values))
            continue
            
        maskaftermax=ds.isel(Sigma_theta=layer).where(ds.isel(Sigma_theta=layer).JULD>=layer_maxT.values)#mask before ox max
        maskduringconsumption=maskaftermax.where(maskaftermax.JULD<=layer_minaftermaxT.values) #should be nan after max and before min
        
        OnesLike[:, layer]=maskduringconsumption.bioOxygen_4ncp.where(np.isnan(maskduringconsumption.bioOxygen_4ncp.values)==True, other=1).values
        #bio Oxygen at timesteps
        layer_bioO2=layerconsumption.bioOxygen_4ncp.dropna(dim="JULD").values
        if len(layer_time)==len(layer_bioO2):
            #calc linear regresssion, save out slope coeff. 
            layer_ST[layer]=layerconsumption.Sigma_theta.values
            RLinear_slope[layer], R_R2[layer], R_intercept[layer]=R_LinearRegression(layer_time, layer_bioO2, layerconsumption.Sigma_theta.values, saveout=False)
            R_ProfNum[layer]=len(layerconsumption.JULD.values)
            R_tstep_days[layer]=layerconsumption.JULD.diff(dim="JULD").dt.days.mean().values
        else:
            print("layer time and R mismatch")
            if len(layer_time)-len(layer_bioO2)==1:
                #linearally interpolate over nan if only one missing (due to outlier removal)
                layer_bioO2=layerconsumption.bioOxygen_4ncp.interpolate_na(dim="JULD").values
                print(layer_bioO2, "layer bioO2")
                print(layer_time, "layer time")
                layer_time=layer_time[~np.isnan(layer_bioO2)]
                layer_bioO2=layer_bioO2[~np.isnan(layer_bioO2)]
                #calc linear regresssion, save plots, save out slope coeff. 
                layer_ST[layer]=layerconsumption.Sigma_theta.values
                RLinear_slope[layer], R_R2[layer], R_intercept[layer]=R_LinearRegression(layer_time, layer_bioO2, layerconsumption.Sigma_theta.values, saveout=False)
                R_ProfNum[layer]=len(layerconsumption.JULD.values)
                R_tstep_days[layer]=layerconsumption.JULD.diff(dim="JULD").dt.days.mean().values
            else:
                #remove layer from calc-cant do regression with nan
                RLinear_slope[layer], R_R2[layer],R_intercept[layer]=np.nan, np.nan, np.nan
                layer_ST[layer]=ds.isel(Sigma_theta=layer).Sigma_theta.values
                OnesLike[:, layer]=np.repeat(np.nan, len(ds.JULD.values))
                R_tstep_days[layer]=np.nan
                continue
    
    
    ones=xr.DataArray(data=OnesLike, dims=["JULD","Sigma_theta"], coords={"JULD": ds.JULD, "Sigma_theta": layer_ST})
    ds["R_linear"]=xr.DataArray(data=RLinear_slope, dims="Sigma_theta")#, coords={"Sigma_theta": layer_ST})
    ds["R_intercept"]=xr.DataArray(data=R_intercept, dims="Sigma_theta")#, coords={"Sigma_theta": layer_ST})
    ds["R_matrix"]=ones*ds.R_linear
    ds["Height"]=ones*ds.LayerThickness
    ds["R_rsqrd"]=xr.DataArray(data=R_R2, dims="Sigma_theta", coords={"Sigma_theta": layer_ST})
    ds["R_Profnum"]=xr.DataArray(data=R_ProfNum, dims="Sigma_theta", coords={"Sigma_theta": layer_ST})
    ds["R_tstep_days"]=xr.DataArray(data=R_tstep_days, dims="Sigma_theta", coords={"Sigma_theta": layer_ST})
    ds["NCPSum_total"]=(ds.ThicknessWeightedR*-.69*ds.Height.sum(dim="Sigma_theta")).sum()
    mld_tend=ds.sel(JULD=str(ds.JULD[-1].dt.year.values)).MLD.max().values
    ds_2MLDmax_tend=ds.where(ds.LayerDepth<=mld_tend, drop=True)
    ds_MLDmax_tend2last=ds.where(ds.LayerDepth>mld_tend, drop=True)
    ds["firstMLDmax_date"]=ds.JULD[0].values
    ds["lastMLDmax_date"]=ds.JULD[-1].values
    ds["ThicknessWeightedR_below_MLDmax_tend"]=((ds_MLDmax_tend2last.R_matrix*ds_MLDmax_tend2last.Height)/ds_MLDmax_tend2last.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
    ds["ThicknessWeightedR_above_MLDmax_tend"]=((ds_2MLDmax_tend.R_matrix*ds_2MLDmax_tend.Height)/ds_2MLDmax_tend.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
    ds["NCPSum_2mldmax_tend"]=(((ds_2MLDmax_tend.R_matrix*ds_2MLDmax_tend.Height)/ds_2MLDmax_tend.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")*-.69*ds_2MLDmax_tend.Height.sum(dim="Sigma_theta")).sum()
    ds["NCPSum_mldmax_tend2deepest"]=(((ds_MLDmax_tend2last.R_matrix*ds_MLDmax_tend2last.Height)/ds_MLDmax_tend2last.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")*-.69*ds_MLDmax_tend2last.Height.sum(dim="Sigma_theta")).sum()
    return ds

### selecting float years (MLD max to MLD max), estimate R, save out files 
(while you don't need to save out files, I found it helpful when working with a large number of float years, as randoms errors will sometimes come up and it's nice to not have to start over)


In [ ]:
## example code to run through all float years that pass prelimiarly lat/lon criteria and have a full annual cycle and save out files with respiration + ANCP + RRC

def select_seasons(ds, kz, subML_mask, smooth_bioO2=True, saveout=True):
    season14=slice("2014-07", "2015-11")
    season15=slice("2015-07", "2016-11")
    season16=slice("2016-07", "2017-11")
    season17=slice("2017-07", "2018-11")
    season18=slice("2018-07", "2019-11")
    season19=slice("2019-07", "2020-11")
    season20=slice("2020-07", "2021-11")
    season21=slice("2021-07", "2022-11")
    season22=slice("2022-07", "2023-11")
    season_list=[season14, season15, season16, season17, season18, season19, season20, season21, season22]
    
    for season in season_list:
        
        ds_season=ds.sel(JULD=season)
        if ds_season.Oxygen.mean(dim="Sigma_theta").count()>25:
            seasonID=str(ds_season.Float_ID.values)[0:7]+"_"+str(ds_season.JULD[0].dt.year.values)
            if os.path.exists("optional path to saved out files"+seasonID+".nc"):
                continue
            yr1=str(ds_season.JULD[0].dt.year.values)
            yr2=str(ds_season.JULD[-1].dt.year.values)
            if yr1==yr2:
                print("only one year")
                continue
            elif np.isnan(ds_season.MLD.sel(JULD=yr1).values).all()==True or np.isnan(ds_season.MLD.sel(JULD=yr2).values).all()==True:
                continue
                #print("empty MLD") - a few times bad/missing salinity causes this
            elif (ds_season.JULD.dropna(dim="JULD").count()-ds_season.Oxygen.mean(dim="Sigma_theta").dropna(dim="JULD").count())>2:
                print("too many missing timesteps in this season")
                continue
            else:
                MLDmax1=ds_season.JULD.where(ds_season.MLD.sel(JULD=yr1)==ds_season.MLD.sel(JULD=yr1).max().values,drop=True)[0].values
                MLDmax2=ds_season.JULD.where(ds_season.sel(JULD=yr2).MLD==ds_season.sel(JULD=yr2).MLD.max().values,drop=True)[0].values

                ds_max2max=ds.sel(JULD=slice(MLDmax1, MLDmax2 + np.timedelta64(1, 'D'))) 
                
                ds_out=O2Fluxes_v6(ds_max2max, kz, subML_mask, nitrate=False)
                #remove layers deeper than Xm (in manuscript, ANCP evaluated at 600m)
                ds_out_X=ds_out.where(ds_out.LayerDepth<X, drop=True)
                #check for large layer S or depth changes 
                s_2nd, znd=S_Z_check_v3(ds_out_X, s_percent=.25, z_percent=.17)
                if np.any([s_2nd, znd])==True:
                    continue
                else: 
                    print(seasonID, "pass z, s check")
                ## get where more than 2 points for LR
                DS_morethan2timesteps=ds_out.where(ds_out.bioOxygen_pressure.count(dim="JULD")>2, drop=True)

                if saveout==False:
                    continue
                else:
                    ds_with_NCP=R_and_NCP_v3(DS_morethan2timesteps, smooth_bioO2=True, saveplots=False)
                    ds_with_NCP.to_netcdf("optional path to save out files"+seasonID+".nc")
                
             

## to get respiraiton curves

In [ ]:
### to get respiraiton curves - calculate the thickness weighted repiration in given depth bins
## Check if layer depth is valid (meaning layer was below mixed layer long enough that oxygen in this layer was checked for decrease)
## Because not all layer depths are below mixed layer for all float year (ie if shallowest MLD is >=50m)
## example with SIZ 

#SIZ_flt_list= get list of flt years in whatever region you choods 

selected_variables=["Lat", "Lon", "MLD", "Height", "R_matrix", "R_linear", "LayerDepth", "R_tstep_days"]

DF_zbinTWR=pd.DataFrame()

for flt in range(len(path_to_saved_out_files)):
    print(flt)
    
    ds_flt=xr.open_dataset(path_to_saved_out_files[flt])
    if str(ds_flt.seasonID.values) in SIZ_flt_list:
        ds_flt_sel=ds_flt[selected_variables]

        ds_flt_sel["R_matrix_daily"]=ds_flt_sel.R_matrix/ds_flt_sel.R_tstep_days  #daily so units can be per day 
        output_ds_SigTheta=xr.Dataset()
        max_depth=ds_flt_sel.LayerDepth.max().values
        min_depth=ds_flt_sel.LayerDepth.min().values
        print(max_depth, "max_depth")
        if min_depth<50:
            test_50=ds_flt_sel.where(ds_flt_sel.LayerDepth<=50, drop=True)
            if test_50.R_matrix_daily.count()==0 and test_50.LayerDepth.count(dim="JULD").max()>2:
                print("no days with resp")
                print(test_50.LayerDepth.count(dim="JULD"))
                ds_flt_sel["twr_50"]=0
            else:
                ds_flt_sel["twr_50"]=((test_50.R_matrix_daily*test_50.Height)/test_50.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_50"]=ds_flt_sel["twr_50"].where(ds_flt_sel["twr_50"]<0, drop=False)
        else: ds_flt_sel["twr_50"]=np.nan

        if min_depth<100 and ds_flt_sel.where(ds_flt_sel.LayerDepth>50, drop=True).where(ds_flt_sel.LayerDepth<=100, drop=True).LayerDepth.count()>0:
            test_50_100=ds_flt_sel.where(ds_flt_sel.LayerDepth>50, drop=True).where(ds_flt_sel.LayerDepth<=100, drop=True)
            if test_50_100.R_matrix_daily.count()==0 and test_50_100.LayerDepth.count(dim="JULD").max()>2:
                ds_flt_sel["twr_50_100"]=0
            else: 
                ds_flt_sel["twr_50_100"]=((test_50_100.R_matrix_daily*test_50_100.Height)/test_50_100.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_50_100"]=ds_flt_sel["twr_50_100"].where(ds_flt_sel["twr_50_100"]<0, drop=False)
        else: ds_flt_sel["twr_50_100"]=np.nan

        test_100_150=ds_flt_sel.where(ds_flt_sel.LayerDepth>100, drop=True).where(ds_flt_sel.LayerDepth<=150, drop=True)
        if test_100_150.R_matrix_daily.count()==0 and test_100_150.LayerDepth.count(dim="JULD").max()>2:
            ds_flt_sel["twr_100_150"]=0
        else: 
            ds_flt_sel["twr_100_150"]=((test_100_150.R_matrix_daily*test_100_150.Height)/test_100_150.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
            ds_flt_sel["twr_100_150"]=ds_flt_sel["twr_100_150"].where(ds_flt_sel["twr_100_150"]<0, drop=False)

        test_150_200=ds_flt_sel.where(ds_flt_sel.LayerDepth>150, drop=True).where(ds_flt_sel.LayerDepth<=200, drop=True)
        if test_150_200.R_matrix_daily.count()==0 and test_150_200.LayerDepth.count(dim="JULD").max()>2:
            ds_flt_sel["twr_150_200"]=0
        else: 
            ds_flt_sel["twr_150_200"]=((test_150_200.R_matrix_daily*test_150_200.Height)/test_150_200.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
            ds_flt_sel["twr_150_200"]=ds_flt_sel["twr_150_200"].where(ds_flt_sel["twr_150_200"]<0, drop=False)

        if max_depth>200:
            test_200_300=ds_flt_sel.where(ds_flt_sel.LayerDepth>200, drop=True).where(ds_flt_sel.LayerDepth<=300, drop=True)
            if test_200_300.R_matrix_daily.count()==0 and test_200_300.LayerDepth.count(dim="JULD").max()>2:
                ds_flt_sel["twr_200_300"]=0
            else: 
                ds_flt_sel["twr_200_300"]=((test_200_300.R_matrix_daily*test_200_300.Height)/test_200_300.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_200_300"]=ds_flt_sel["twr_200_300"].where(ds_flt_sel["twr_200_300"]<0, drop=False)
        else: ds_flt_sel["twr_200_300"]=np.nan

        if max_depth>300 and ds_flt_sel.where(ds_flt_sel.LayerDepth>300, drop=True).where(ds_flt_sel.LayerDepth<=400, drop=True).LayerDepth.count()>0:
            test_300_400=ds_flt_sel.where(ds_flt_sel.LayerDepth>300, drop=True).where(ds_flt_sel.LayerDepth<=400, drop=True)
            if test_300_400.R_matrix_daily.count()==0 and test_300_400.LayerDepth.count(dim="JULD").max()>2:
                ds_flt_sel["twr_300_400"]=0
            else: 
                ds_flt_sel["twr_300_400"]=((test_300_400.R_matrix_daily*test_300_400.Height)/test_300_400.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_300_400"]=ds_flt_sel["twr_300_400"].where(ds_flt_sel["twr_300_400"]<0, drop=False)
        else: ds_flt_sel["twr_300_400"]=np.nan

        if max_depth>400 and ds_flt_sel.where(ds_flt_sel.LayerDepth>400, drop=True).where(ds_flt_sel.LayerDepth<=500, drop=True).LayerDepth.count()>0:
            test_400_500=ds_flt_sel.where(ds_flt_sel.LayerDepth>400, drop=True).where(ds_flt_sel.LayerDepth<=500, drop=True)
            print(test_400_500.LayerDepth)
            if test_400_500.R_matrix_daily.count()==0 and test_400_500.LayerDepth.count(dim="JULD").max()>2:
                ds_flt_sel["twr_400_500"]=0
            else: 
                ds_flt_sel["twr_400_500"]=((test_400_500.R_matrix_daily*test_400_500.Height)/test_400_500.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_400_500"]=ds_flt_sel["twr_400_500"].where(ds_flt_sel["twr_400_500"]<0, drop=False)
        else: ds_flt_sel["twr_400_500"]=np.nan

        if max_depth>500 and ds_flt_sel.where(ds_flt_sel.LayerDepth>500, drop=True).where(ds_flt_sel.LayerDepth<=600, drop=True).LayerDepth.count()>0:
            test_500_600=ds_flt_sel.where(ds_flt_sel.LayerDepth>500, drop=True).where(ds_flt_sel.LayerDepth<=600, drop=True)
            if test_500_600.R_matrix_daily.count()==0 and test_500_600.LayerDepth.count(dim="JULD").max()>2:
                ds_flt_sel["twr_500_600"]=0
            else: 
                ds_flt_sel["twr_500_600"]=((test_500_600.R_matrix_daily*test_500_600.Height)/test_500_600.Height.sum(dim="Sigma_theta")).sum(dim="Sigma_theta")
                ds_flt_sel["twr_500_600"]=ds_flt_sel["twr_500_600"].where(ds_flt_sel["twr_500_600"]<0, drop=False)
        else: ds_flt_sel["twr_500_600"]=np.nan


        twr_molm3=np.zeros([len(ds_flt_sel.JULD.values), len([25, 75, 125, 175, 250,350,450,550])])
        #UNITS [mol O2 m^-3 day^-1]
        twr_molm3[:, 0]=ds_flt_sel.twr_50.values 
        twr_molm3[:, 1]=ds_flt_sel.twr_50_100.values
        twr_molm3[:, 2]=ds_flt_sel.twr_100_150.values
        twr_molm3[:, 3]=ds_flt_sel.twr_150_200.values
        twr_molm3[:, 4]=ds_flt_sel.twr_200_300.values
        twr_molm3[:, 5]=ds_flt_sel.twr_300_400.values
        twr_molm3[:, 6]=ds_flt_sel.twr_400_500.values
        twr_molm3[:, 7]=ds_flt_sel.twr_500_600.values

        ds_flt_sel["twr_molm3"]=xr.DataArray(twr_molm3, dims=["JULD", "depth_center"],coords=({"JULD":ds_flt_sel.JULD.values, "depth_center":[25, 75, 125, 175, 250 ,350,450,550]}))
        output_ds_SigTheta["twr_molm3_meanall"]=ds_flt_sel.twr_molm3.mean(dim="JULD") 
        output_df=output_ds_SigTheta.to_dataframe()
        DF_zbinTWR=DF_zbinTWR.append(output_df)


## get EP from NPP 
can loop through datasets with all float years or float seasons, call functions based on float year year_slice, jan Lat, Lon. 

In [ ]:
def EP_calc(year_slice, lat_in, lon_in, NPP_file_list, NPP_model, missing_apr22, year_start):
    """SST IS NOAA High-resolution Blended Analysis of Daily SST 
    Silica concentration is WOA monthly climatology
    """
    lon_in_360=convert_180_lon_to_360(lon_in)#so_sst has lon in 0-360 instead of -180 to 180, sst 2014 is -180 to 180 
    if year_start==2014:
        sst_match=SO_SST_14.sel(lat=lat_in, lon=lon_in, method="nearest").sel(time=year_slice)
    else:
        sst_match=SO_SST.sel(lat=lat_in, lon=lon_in_360, method="nearest").sel(time=year_slice)
    print(sst_match.time.values)
    print(sst_match.sst.values)
    # check nothing is funky
    if abs(lat_in-sst_match.lat)>0.5 or abs(lon_in_360-sst_match.lon)>0.5:
        print("EXIT-LON LAT match for SST IS OFF")
        print(lon_in_360, sst_match.lon, lat_in, sst_match.lat)
    NPP=open_and_process_raster_forEP(NPP_file_list, NPP_model, lat_in, lon_in, missing_apr22)
    NPP_df=pd.DataFrame.from_dict(NPP, orient='index')
    NPP_df.columns = ['npp']
    NPP_match_flt=NPP_df.to_xarray().npp.drop("index")
    NPP_match_flt=NPP_match_flt.assign_coords({"index": sst_match.time.values}).rename({"index": "time"})
    print(NPP_match_flt.time, "Npp time")
    SI=xr.DataArray(SI_match(lat_in, lon_in), coords={"time": sst_match.time})
    #convert Si from umol/kg to mmol/m3, NPP to mmolC/m2 day for Britten :
    SI=SI/1000*1027
    E_ratio_Britten=(3.72-0.16*sst_match.sst-0.04*SI)*(NPP_match_flt/12)**(-0.55)
    EP_Britten=(NPP_match_flt/12)*E_ratio_Britten
    #print(EP_Britten)
    EP_Britten_masked=EP_Britten.where(E_ratio_Britten<1, drop=False)#following methods, see https://agupubs.onlinelibrary.wiley.com/doi/10.1002/2018JC013787 
    #print(EP_Britten_masked)
    EP_Laws_eq2=NPP_match_flt.values*(((0.5857-0.0165*sst_match.sst)*NPP_match_flt.values)/(51.7+NPP_match_flt.values))
    EP_Laws_eq3=NPP_match_flt.values*(0.04756*(0.78-(.43*sst_match.sst)/30)*NPP_match_flt.values**(0.307))
    EP_Li_Cassar=(8.57*NPP_match_flt.values)/(17.9+sst_match.sst)
    
    return NPP_match_flt, EP_Laws_eq2, EP_Laws_eq3, EP_Li_Cassar, EP_Britten_masked



"""
Please check the file formatting/ file names for the NPP files from https://orca.science.oregonstate.edu/index.php
in case they update you should adjust the reading + processing part accordingly!

Note for processing files from documentation : 
For 1080 by 2160 data, the grid spacing is 1/6 of a degree in both latitude and longitude.
1080 rows * 1/6 degree per row = 180 degrees of latitude (+90 to -90).
2160 columns * 1/6 degree per column = 360 degrees of longitude (-180 to +180).
The north west corner of the start of the hdf file is at +90 lat, -180 lon.
To obtain the location of the center of any pixel:
take the number of rows and columns you are away from the NW corner,
multiply by the grid spacing to get the change in latitude and longitude,
subtract the change in latitude from +90 lat,
add the change in longitude to -180 lon;
shift the latitude down (subtract) by 1/2 of a grid spacing
and shift the longitude over (add) by 1/2 of a grid spacing"""


#because in S Hemi, winter to winter annual cycle spans two years. 
Jul_doy=["182", '183']
Aug_doy=['213',"214"]
Sep_doy=['244','245']
Oct_doy=["274", "275"]
Nov_doy=["305", "306"]
Dec_doy=["335", "336"]
Jan_doy=["001"]
Feb_doy=["032"]
Mar_doy=["060", "061"]
Arp_doy=["091", "092"]
May_doy=["121", "122"]
Jun_doy=["152", "153"]


def open_and_process_raster_forEP(file_list, label, lat_in, lon_in, missing_april22=False):
        NPP_dict = {}
        if missing_april22==True:
            for i, file in enumerate(file_list):
                NPP = rxr.open_rasterio(file, masked=True)[0]
                NPP['lat'] = (90 - (NPP.y - NPP.y[0]) * (1/6)) - (1/2 * 1/6)
                NPP["lon"] = (-180 + (NPP.x - NPP.x[0]) * (1/6)) + (1/2 * 1/6)
                if abs(lat_in-NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").y)>0.2 or abs(lon_in-NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").x)>0.2:
                    print("Break-LON LAT match for EP IS OFF")
                    break
                NPP = NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").values
                if NPP==-9999.:
                    NPP=np.nan
                print(file[-7:-4])
                if file[-7:-4] in Jul_doy:
                    NPP_dict[f'NPP{1}_{label}'] = NPP
                elif file[-7:-4] in Aug_doy:
                    NPP_dict[f'NPP{2}_{label}'] = NPP
                elif file[-7:-4] in Sep_doy:
                    NPP_dict[f'NPP{3}_{label}'] = NPP
                elif file[-7:-4] in Oct_doy:
                    NPP_dict[f'NPP{4}_{label}'] = NPP
                elif file[-7:-4] in Nov_doy:
                    NPP_dict[f'NPP{5}_{label}'] = NPP
                elif file[-7:-4] in Dec_doy:
                    NPP_dict[f'NPP{6}_{label}'] = NPP
                elif file[-7:-4] in Jan_doy:
                    NPP_dict[f'NPP{7}_{label}'] = NPP
                elif file[-7:-4] in Feb_doy:
                    NPP_dict[f'NPP{8}_{label}'] = NPP
                elif file[-7:-4] in Mar_doy:
                    NPP_dict[f'NPP{9}_{label}'] = NPP
                elif file[-7:-4] in May_doy:
                    NPP_dict[f'NPP{11}_{label}'] = NPP
                elif file[-7:-4] in Jun_doy:
                    NPP_dict[f'NPP{12}_{label}'] = NPP
            NPP_dict[f'NPP{10}_{label}']=np.nan #because no april file available
            print(len(NPP_dict))
            
        else:      
            for i, file in enumerate(file_list):
                print(file[-10:-1])
                NPP = rxr.open_rasterio(file, masked=True)[0]
                NPP['lat'] = (90 - (NPP.y - NPP.y[0]) * (1/6)) - (1/2 * 1/6)
                NPP["lon"] = (-180 + (NPP.x - NPP.x[0]) * (1/6)) + (1/2 * 1/6)
                if abs(lat_in-NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").y)>0.2 or abs(lon_in-NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").x)>0.2:
                    print("Break-LON LAT match for EP IS OFF")
                    break
                NPP = NPP.set_index({"x": "lon", "y": "lat"}).sel(y=lat_in, x=lon_in, method="nearest").values

                if NPP==-9999.:
                    NPP=np.nan
                NPP_dict[f'NPP{i+1}_{label}'] = NPP
        if len(NPP_dict)<12:
            print("warning, Npp dict does not have 12 months")
        return NPP_dict
    


### Make paper figures: 
For figures -I save out datasets generated from the code above to make it easier to make the different figures
they are as follows: 
ds_ANCP_EP = xarray dataset with the following variables: 

"ANCP_600", (ancp)

"RRC",  (respired and re-entrained carbon)

"fResp_600",  (fraction of RRC) 

"ANCP100_600", ('ancp' calculated from 100m to 600m instead of MLDwint to 600m)

"zone", (frontal zone)

"mld_lastmax", (MLDwint)

"Lat_jan", 

"Lon_jan",

as well as matched satellite EP (for example 'Laws3_sum_cafe' is EP from using the Laws equation 3 using Cafe NPP)


ds_bNCP_EP:
has bNCP_ML as well as Jan lat, lon, and EP matches. 


## Figure 1

In [ ]:
### here read in the isopycnal gridded 5905994 
fig, ax = plt.subplots()
color_var=ds.Sigma_theta
cmap = plt.cm.magma_r  
norm = Normalize(27.54, 27.83)
#print(color_var.min(), color_var.max())
for i in range(len(ds.Sigma_theta.values)):
    x=np.arange(0,len(ds.isel(Sigma_theta=i).Height.dropna(dim="JULD").JULD.values))
    z_color = cmap(norm(color_var[i]))
    y=(ds.isel(Sigma_theta=i).R_linear/(ds.isel(Sigma_theta=i).Sigma_theta+1000)).values*(10**6)*x+(ds.isel(Sigma_theta=i).R_intercept/(ds.isel(Sigma_theta=i).Sigma_theta+1000)*10**6).values
    ax.plot(ds.isel(Sigma_theta=i).Height.dropna(dim="JULD").JULD.values, y, color="lightblue")
    ds["bio4ncp_umolkg"]=ds.bioOxygen_4ncp/(ds.isel(Sigma_theta=i).Sigma_theta+1000)*10**6
    ds.isel(Sigma_theta=i).plot.scatter(ax=ax, x="JULD", y="bio4ncp_umolkg", color=z_color, s=68)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([]) 
cbar = fig.colorbar(sm, ax=ax) 
cbar.ax.tick_params(labelsize=17) 
# invert so the colors make senes:  lighter colors = lighter waters
cbar.ax.invert_yaxis()
cbar.set_ticklabels([f"{tick:.2f}" for tick in cbar.get_ticks()])
cbar.set_label('potential density (kg m$^{-3}$)', fontsize=18)
ax.set_title("")
ax.tick_params(labelsize=15)
year_month = np.array([ts[:7] for ts in ds.Height.JULD.astype(str).values[0:None:8]])
ax.set_xticks(year_month)
ax.set_xticklabels(year_month, rotation=45, fontsize=13)
ax.set_ylabel("Oxygen (umol kg$^{-3}$)", fontsize=18)
ax.set_xlabel("")
fig.tight_layout()
plt.show()


## Figure 3 


In [ ]:
fig, ax = plt.subplots(figsize=(7,8))
## merged ds has only float seasons/years that have both bNCP_ML and ANCP (find where ID is in both)

im=merged_ds.plot.scatter(ax=ax, x="ANCP_600", y="bNCP_ML", hue="fResp_600", s=60, alpha=0.9, add_guide=False)
ax.set_ylabel("bNCP$_{ML}$ \n (mol C m$^{-2}$ season$^{-1}$)", fontsize=19)
ax.set_xlabel("ANCP (mol C m$^{-2}$ year$^{-1}$)", fontsize=19)
cb = plt.colorbar(im,orientation="horizontal", shrink=.8)
cb.set_label(label='fRRC',size=18)
cb.ax.tick_params(labelsize=15)
ax.set_yticks([0,1,2,3,4,5,6], fontsize=18)
ax.set_xticks([0,1,2,3,4,5,6],fontsize=18)
ax.plot([0,6.9], [0,6.9])
ax.plot([0,6.9], [0,6.9])
ax.set_xlim(0,6.9)
ax.set_ylim(0,6.9)



In [ ]:
fig, axb =plt.subplots(figsize=(9,7))

#b)
df_1002deepest = ds_ANCP_EP.to_dataframe()[['zone', 'ANCP100_600']]
df_mldmax2deepest_tend = ds_ANCP_EP.to_dataframe()[['zone', 'ANCP_600']]
df_1002deepest['variable'] = 'ANCP 100-600m'
df_mldmax2deepest_tend['variable'] = 'ANCP'
df_1002deepest = df_1002deepest.rename(columns={'ANCP100_600': 'value'})
df_mldmax2deepest_tend = df_mldmax2deepest_tend.rename(columns={'ANCP_600': 'value'})
combined_df = pd.concat([df_1002deepest, df_mldmax2deepest_tend])
# List of zones to include
zones_to_include = [ "SIZ","AZ", "SAZ", "STZ"]
filtered_df = combined_df[combined_df['zone'].isin(zones_to_include)]
# Create a box plot
color_palette = sns.color_palette("colorblind")
sns.boxplot(ax=axb, x='zone', y='value', hue='variable', data=filtered_df.reset_index(), palette=color_palette, order=["SIZ","AZ", "SAZ", "STZ"])#, boxprops=dict(alpha=.8))
# Set plot labels and title
axb.set_xlabel('Zone', fontsize=20)
axb.set_ylabel('mol C m$^{-2}$ year$^{-1}$', fontsize=20)
axb.tick_params(axis='both', which='major', labelsize=18)
axa.text(-0.1, 1.1, 'A)', transform=axa.transAxes, fontsize=14, va='top', ha='right')
axb.legend(fontsize=18)

axb.text(1.2, 1.1, 'B)', transform=axa.transAxes, fontsize=14, va='top', ha='right')
axb.legend(fontsize=18)



In [ ]:

## Fig with MAP

stf_lat_xr=stf.Lat.to_xarray()
stf_lon_xr=stf.Lon.to_xarray()

saf_lat_xr=saf.Lat.to_xarray()
saf_lon_xr=saf.Lon.to_xarray()

pf_lat_xr=pf.Lat.to_xarray()
pf_lon_xr=pf.Lon.to_xarray()

siz_lat_xr=siz.Lat.to_xarray()
siz_lon_xr=siz.Lon.to_xarray()

df_ax=ds_ANCP_EP.to_dataframe()
fig, axis = plt.subplots(
    1, 1, subplot_kw=dict(projection=ccrs.Stereographic(central_latitude=-90)), figsize=(10,8), dpi=300
)

sns.scatterplot(ax=axis, transform=ccrs.PlateCarree(), x=df_ax['Lon_jan'], y=df_ax['Lat_jan'], hue=df_ax["ANCP_600"],palette="viridis", s=42, hue_norm=(0,4),edgecolor="black", legend=False)
sns.scatterplot(ax=axis, transform=ccrs.PlateCarree(), x=ds_bNCP_EP['Jan_lon'], y=ds_bNCP_EP['Jan_lat'], hue=ds_bNCP_EP["bNCP_ML"],s=42, palette="viridis", hue_norm=(0,4),marker="^", edgecolor="black", legend=False)

stf_lon_forplot=stf_lon_xr.where(stf_lat_xr>-90).where(stf_lat_xr<-30)
stf_lat_forplot=stf_lat_xr.where(stf_lat_xr>-90).where(stf_lat_xr<-30)
axis.plot(stf_lon_forplot.values, stf_lat_forplot.values, transform=ccrs.PlateCarree(), color="darkorange", linewidth=2, alpha=0.7)
axis.plot(saf_lon_xr.where(saf_lat_xr>-90).values, saf_lat_xr.where(saf_lat_xr>-90).values, transform=ccrs.PlateCarree(), linewidth=2, color="darkmagenta", alpha=0.7)
axis.plot(pf_lon_xr.where(pf_lat_xr>-90).values, pf_lat_xr.where(pf_lat_xr>-90).values, transform=ccrs.PlateCarree(), color="grey", linewidth=2, alpha=0.7)
axis.plot(siz_lon_xr.where(siz_lat_xr>-90).values, siz_lat_xr.where(siz_lat_xr>-90).values, transform=ccrs.PlateCarree(), color="red",linewidth=2,  alpha=0.7)


axis.coastlines(zorder=10)
axis.add_feature(cfeature.LAND, zorder=11)

norm = plt.Normalize(0, 4)#to get correct colorbar
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

gls1 = axis.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, \
                xlocs=range(-180,171,60), ylocs=[], \
                color='gray', alpha=0.5, linestyle='--', zorder=15)
gls2 = axis.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, \
                xlocs=[], ylocs=range(-70, -30, 10), \
                color='gray', alpha=0.5, linestyle='--', zorder=15)

# declare text size
XTEXT_SIZE = 11
YTEXT_SIZE = 11
# I want to rotate text at bottom edge, ...
# for text justification: 'ha':'right' is used to avoid overlapping with the map awkwardly
gls1.xlabel_style = {'size': XTEXT_SIZE}
gls1.ylabel_style = {'size':YTEXT_SIZE}
gls2.xlabel_style = {'size': XTEXT_SIZE}
gls2.ylabel_style = {'size':YTEXT_SIZE}

gls2.ylabel_style = {'rotation': 45}

axis.set_extent([-180, 180, -90, -35], ccrs.PlateCarree())
# add a circle border 
theta = np.linspace(0, 2*np.pi, 100)
center, radius = [0.5, 0.5], 0.5
verts = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)

axis.set_boundary(circle, transform=axis.transAxes)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markeredgecolor='black', linewidth=2, label='ANCP \n (mol C m$^{-2}$ year$^{-1}$)'),
    Line2D([0], [0], marker='^',  color='w',markeredgecolor='black', label='Mixed Layer bNCP \n (mol C m$^{-2}$ season$^{-1}$')
]

axis.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1.1), fontsize=10)

#colorbar
cbar=fig.colorbar(sm, ax=axis, shrink=0.35, location='bottom',pad=.09,  extend='max')
cbar.ax.tick_params(labelsize=16)
cbar.set_label('mol C m$^{-2}$ year$^{-1}$', fontsize=16)

fig.tight_layout()


In [ ]:
### read in POC flux data
POC_flux="path_to_file"
POC_flux_ds=xr.open_dataset(POC_flux)
POC_flux_ds["POC_large_concentration"]=POC_flux_ds.POC_large_flux/75
POC_flux_ds["POC_small_concentration"]=POC_flux_ds.POC_small_flux/2.5
POC_flux_ds["POC_concentration"]=POC_flux_ds.POC_small_concentration+POC_flux_ds.POC_large_concentration


Just_SO=POC_flux_ds.sel(lat=slice(-75,-30))


Just_SO_zone=np.zeros([len(Just_SO.lon.values), len(Just_SO.lat.values)])

stf=pd.read_csv('path to Gray Frontal positions-see methods for ref') 
saf=pd.read_csv("")
siz=pd.read_csv('') 
pf=pd.read_csv('')
fronts = loadmat('')

stf_poly_gray = Polygon(zip(stf.Lon, stf.Lat))
stf_gfd_gray=gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[stf_poly_gray])

pf_poly_gray = Polygon(zip(pf.Lon, pf.Lat))
pf_gfd_gray=gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[pf_poly_gray])

saf_poly_gray = Polygon(zip(saf.Lon, saf.Lat))
saf_gfd_gray=gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[saf_poly_gray])

siz_poly_gray = Polygon(zip(siz.Lon, siz.Lat))
siz_gfd_gray=gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[siz_poly_gray])



for x in range(len(Just_SO.lon.values)):
    for y in range(len(Just_SO.lat.values)):
        print(Just_SO.isel(lon=x, lat=y)["lon"].values, Just_SO.isel(lon=x, lat=y)["lat"].values)
        coords=Point(Just_SO.isel(lon=x, lat=y)["lon"].values, Just_SO.isel(lon=x, lat=y)["lat"].values)
        prof_coords = gpd.GeoSeries(coords, crs="4326")

        if siz_gfd_gray.contains(prof_coords)[0]==True:
            Just_SO_zone[x, y]=10 #"SOZ"
        elif pf_gfd_gray.contains(prof_coords)[0]==True:
            Just_SO_zone[x, y]=20 #"AZ
        elif saf_gfd_gray.contains(prof_coords)[0]==True:
            Just_SO_zone[x, y]=30 #"PFZ"
        elif stf_gfd_gray.contains(prof_coords)[0]==True:
            Just_SO_zone[x, y]=50  #"STZ
        else: Just_SO_zone[x, y]=40#"SAZ"


Just_SO_zone_DA=xr.DataArray(Just_SO_zone, dims=["lon", "lat"])
Just_SO_zone_DA=Just_SO_zone_DA.assign_coords({"lon":Just_SO.lon.values, "lat":Just_SO.lat.values})

SIZ_POC_flux=Just_SO.where(Just_SO_zone_DA==10, drop=True)
AZ_POC_flux=Just_SO.where(Just_SO_zone_DA==20, drop=True)
PFZ_POC_flux=Just_SO.where(Just_SO_zone_DA==30, drop=True)
SAZ_POC_flux=Just_SO.where(Just_SO_zone_DA==40, drop=True)
STZ_POC_flux=Just_SO.where(Just_SO_zone_DA==50, drop=True)

#SIZ, AZ, SAZ, PFZ, STZ POC loss
#first remove above 10 m and below 999m
SIZ_POC_Flux_sliced=SIZ_POC_flux.POC_total_flux.sel(depth=slice(10,999))
AZ_POC_Flux_sliced=AZ_POC_flux.POC_total_flux.sel(depth=slice(10,999))
SAZ_POC_Flux_sliced=SAZ_POC_flux.POC_total_flux.sel(depth=slice(10,999))
STZ_POC_Flux_sliced=STZ_POC_flux.POC_total_flux.sel(depth=slice(10,999))

SIZ_POC_conc_sliced=SIZ_POC_flux.POC_concentration.sel(depth=slice(10,999))
AZ_POC_conc_sliced=AZ_POC_flux.POC_concentration.sel(depth=slice(10,999))
SAZ_POC_conc_sliced=SAZ_POC_flux.POC_concentration.sel(depth=slice(10,999))
STZ_POC_conc_sliced=STZ_POC_flux.POC_concentration.sel(depth=slice(10,999))

#calculated POC loss, make coordinates centered appropriately
SIZ_POC_loss=SIZ_POC_Flux_sliced.diff(dim="depth")/(SIZ_POC_Flux_sliced.depth.diff(dim="depth"))
SIZ_POC_loss_coords=SIZ_POC_loss.assign_coords({"depth":SIZ_POC_Flux_sliced.depth.diff(dim="depth", label="lower")/2+SIZ_POC_Flux_sliced.depth}).rename({"depth":"depth_center"})
SIZ_POC_loss_mean=SIZ_POC_loss_coords.mean(dim=["lat", "lon", "month"])

AZ_POC_loss=AZ_POC_Flux_sliced.diff(dim="depth")/(AZ_POC_Flux_sliced.depth.diff(dim="depth"))
AZ_POC_loss_coords=AZ_POC_loss.assign_coords({"depth":AZ_POC_Flux_sliced.depth.diff(dim="depth", label="lower")/2+AZ_POC_Flux_sliced.depth}).rename({"depth":"depth_center"})
AZ_POC_loss_mean=AZ_POC_loss_coords.mean(dim=["lat", "lon", "month"])

SAZ_POC_loss=SAZ_POC_Flux_sliced.diff(dim="depth")/(SAZ_POC_Flux_sliced.depth.diff(dim="depth"))
SAZ_POC_loss_coords=SAZ_POC_loss.assign_coords({"depth":SAZ_POC_Flux_sliced.depth.diff(dim="depth", label="lower")/2+SAZ_POC_Flux_sliced.depth}).rename({"depth":"depth_center"})
SAZ_POC_loss_mean=SAZ_POC_loss_coords.mean(dim=["lat", "lon", "month"])

STZ_POC_loss=STZ_POC_Flux_sliced.diff(dim="depth")/(STZ_POC_Flux_sliced.depth.diff(dim="depth"))
STZ_POC_loss_coords=STZ_POC_loss.assign_coords({"depth":STZ_POC_Flux_sliced.depth.diff(dim="depth", label="lower")/2+STZ_POC_Flux_sliced.depth}).rename({"depth":"depth_center"})
STZ_POC_loss_mean=STZ_POC_loss_coords.mean(dim=["lat", "lon", "month"])


#C_spec = POC loss divided by POC concentration at the top of the depth interval (as in Iverson et al)
SIZ_C_SPEC=SIZ_POC_loss_coords*(1/SIZ_POC_conc_sliced.isel(depth=slice(0, -1)).values)
SAZ_C_SPEC=SAZ_POC_loss_coords*(1/SAZ_POC_conc_sliced.isel(depth=slice(0, -1)).values)
AZ_C_SPEC=AZ_POC_loss_coords*(1/AZ_POC_conc_sliced.isel(depth=slice(0, -1)).values)
STZ_C_SPEC=STZ_POC_loss_coords*(1/STZ_POC_conc_sliced.isel(depth=slice(0, -1)).values)



In [ ]:
# functions for respiration / poc plots
def calc_ci_POC(da, alpha):
    if "depth" in da.dims:
        da=da.rename({"depth": "depth_center"})
    depth_center_index=da["depth_center"].values 
    ci_upper=[]
    ci_lower=[]
    for k in depth_center_index:
        data= da.sel(depth_center=k)
        flattened_arr=data.values.flatten()
        data_dropna=flattened_arr[~np.isnan(flattened_arr)]
        ci= stats.t.interval(alpha, df=len(data_dropna)-1, loc=np.mean(data_dropna), scale=stats.sem(data_dropna))
        ci_upper.append(ci[0])
        ci_lower.append(ci[-1])
    return(ci_upper, ci_lower)

def calc_CI(s, alpha):
    """
    For 2-D data, find CI along one dimesion (ex for each depth bin)
    s is a series with a varible of interest (ex resp rate) and dim is the index name 
    that the ci will be computed along (ex depth)
    """
    index_list=s.groupby("depth_center").count()["depth_center"].values  #array of monotonic unique index values
    ci_upper=[]
    ci_lower=[]
    for k in index_list:
        print(k)
        data= s.sel(depth_center=k)
        data_dropna=data.dropna(dim="depth_center")
        ci= stats.t.interval(alpha, df=len(data_dropna)-1, loc=np.mean(data_dropna), scale=stats.sem(data_dropna))
        ci_upper.append(ci[0])
        ci_lower.append(ci[-1])
    return(ci_upper, ci_lower)

CB_color_cycle = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

In [ ]:
# see 

In [ ]:
#########POC flux vs respiration, POC loss vs. respiration for SIZ vs SAZ#########
#sorry this is not pretty code 
fig, ((ax2,ax3),(ax, ax1)) = plt.subplots(2,2, figsize=(9,12))
axb=ax.twiny()
ax1b=ax1.twiny()

ax2b=ax2.twiny()
ax3b=ax3.twiny()
#SIZ
((sizR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax, y='depth_center', color=CB_color_cycle[4], lw=3)#label="Float derived respiration"
(SIZ_POC_loss_mean/-12).sel(depth_center=slice(100,600)).plot(ax=axb, y='depth_center', color=CB_color_cycle[0], label="POC loss", lw=3)
(SIZ_POC_loss_mean/-12).sel(depth_center=slice(0,100)).plot(ax=axb, y='depth_center', color=CB_color_cycle[0], linestyle="dashed", alpha=0.6, lw=3)
ci_up, ci_low=calc_CI(sizR["annualResp_O2_molm3"]*-0.69*1000, 0.95)
ci_up_poc, ci_low_poc=calc_ci_POC(SIZ_POC_loss_coords/-12, 0.95)
z=sizR["annualResp_O2_molm3"].groupby("depth_center").count()["depth_center"].values

ax.plot(np.array(ci_up), z, color=CB_color_cycle[4], alpha=0.2)
ax.plot(np.array(ci_low), z, color=CB_color_cycle[4], alpha=0.2)
ax.fill_betweenx(z, np.array(ci_low), np.array(ci_up), where=(np.array(ci_low) >  np.array(ci_up)),  color=CB_color_cycle[4], alpha=0.3)
z_poc=SIZ_POC_loss_coords["depth_center"].values
axb.plot(np.array(ci_up_poc), z_poc, color=CB_color_cycle[0], alpha=0.2)
axb.plot(np.array(ci_low_poc), z_poc, color=CB_color_cycle[0], alpha=0.2)
axb.fill_betweenx(z_poc, np.array(ci_low_poc), np.array(ci_up_poc), where=(np.array(ci_low_poc) >  np.array(ci_up_poc)),  color=CB_color_cycle[0], alpha=0.2)

#SAZ
((sazR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax1, y='depth_center', color=CB_color_cycle[4], lw=3)
(SAZ_POC_loss_mean/-12).sel(depth_center=slice(100,600)).plot(ax=ax1b, y='depth_center', color=CB_color_cycle[0],  lw=3)
(SAZ_POC_loss_mean/-12).sel(depth_center=slice(0,100)).plot(ax=ax1b, y='depth_center', color=CB_color_cycle[0],  lw=3,linestyle="dashed",alpha=0.6)
ci_upN, ci_lowN=calc_CI(sazR["annualResp_O2_molm3"]*1000*-.69, 0.95)
ci_up_pocN, ci_low_pocN=calc_ci_POC(SAZ_POC_loss_coords/-12, 0.95)
ax1.plot(np.array(ci_upN), z, color=CB_color_cycle[4], alpha=0.2)
ax1.plot(np.array(ci_lowN), z, color=CB_color_cycle[4], alpha=0.2)
ax1.fill_betweenx(z, np.array(ci_lowN), np.array(ci_upN), where=(np.array(ci_lowN) >  np.array(ci_upN)),  color=CB_color_cycle[4], alpha=0.2)
ax1b.plot(np.array(ci_up_pocN), z_poc, color=CB_color_cycle[0], alpha=0.2)
ax1b.plot(np.array(ci_low_pocN), z_poc, color=CB_color_cycle[0], alpha=0.2)
ax1b.fill_betweenx(z_poc, np.array(ci_low_pocN), np.array(ci_up_pocN), where=(np.array(ci_low_pocN) >  np.array(ci_up_pocN)),  color=CB_color_cycle[0], alpha=0.2)

## ax2, ax3
SIZ_POC_flux.POC_total_flux.mean(dim=["lat","lon", "month"]).sel(depth=slice(125,700)).plot(ax=ax2b,  y='depth', color="navy", label="POC Flux", lw=3)
SAZ_POC_flux.POC_total_flux.mean(dim=["lat","lon", "month"]).sel(depth=slice(125,700)).plot(ax=ax3b,  y='depth', color="navy", lw=3)
SIZ_POC_flux.POC_total_flux.mean(dim=["lat","lon", "month"]).sel(depth=slice(0,125)).plot(ax=ax2b,  y='depth', color="navy", lw=3, alpha=0.6, linestyle="dashed")
SAZ_POC_flux.POC_total_flux.mean(dim=["lat","lon", "month"]).sel(depth=slice(0,125)).plot(ax=ax3b,  y='depth', color="navy", lw=3, alpha= 0.6, linestyle="dashed")

((sizR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax2, y='depth_center', color=CB_color_cycle[4], lw=3)#label="Float derived respiration"
((sazR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax3, y='depth_center', label="Mean float-derived respiration rate", color=CB_color_cycle[4], lw=3)
((sizR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax2, y='depth_center', color=CB_color_cycle[4], lw=3)#label="Float derived respiration"
((sazR.annualResp_O2_molm3.groupby("depth_center").mean()*1000*-.69)).plot(ax=ax3, y='depth_center', label="Mean float-derived respiration rate", color=CB_color_cycle[4], lw=3)

ci_up_poc_saz, ci_low_poc_saz=calc_ci_POC(SAZ_POC_flux.POC_total_flux, 0.95)
ci_up_poc_siz, ci_low_poc_siz=calc_ci_POC(SIZ_POC_flux.POC_total_flux, 0.95)
z_poc_flux=SIZ_POC_flux.POC_total_flux["depth"].values
ax2b.plot(np.array(ci_up_poc_siz), z_poc_flux, color="navy", alpha=0.3)
ax2b.plot(np.array(ci_low_poc_siz), z_poc_flux, color="navy", alpha=0.3)
ax2b.fill_betweenx(z_poc_flux, np.array(ci_low_poc_siz), np.array(ci_up_poc_siz), where=(np.array(ci_low_poc_siz) >  np.array(ci_up_poc_siz)),  color="navy", alpha=0.2)
ax3b.plot(np.array(ci_up_poc_saz), z_poc_flux, color="navy", alpha=0.3)
ax3b.plot(np.array(ci_low_poc_saz), z_poc_flux, color="navy", alpha=0.3)
ax3b.fill_betweenx(z_poc_flux, np.array(ci_low_poc_saz), np.array(ci_up_poc_saz), where=(np.array(ci_low_poc_saz) >  np.array(ci_up_poc_saz)),  color="navy", alpha=0.2)

ax2.plot(np.array(ci_up), z, color=CB_color_cycle[4], alpha=0.3)
ax2.plot(np.array(ci_low), z, color=CB_color_cycle[4], alpha=0.3)
ax2.fill_betweenx(z, np.array(ci_low), np.array(ci_up), where=(np.array(ci_low) >  np.array(ci_up)),  color=CB_color_cycle[4], alpha=0.2)

ax3.plot(np.array(ci_upN), z, color=CB_color_cycle[4], alpha=0.3)
ax3.plot(np.array(ci_lowN), z, color=CB_color_cycle[4], alpha=0.3)
ax3.fill_betweenx(z, np.array(ci_lowN), np.array(ci_upN), where=(np.array(ci_lowN) >  np.array(ci_upN)),  color=CB_color_cycle[4], alpha=0.2)

ax.set_xlim(0, .35)
ax1.set_xlim(0, .35)
ax.set_ylim(600,20)
ax1.set_ylim(600,20)
axb.set_xlim(0, 0.35)
ax1b.set_xlim(0,0.35)

ax2.set_ylim(600,20)
ax3.set_ylim(600,20)
ax2.set_xlim(0, .35)
ax3.set_xlim(0, .35)
ax2b.set_xlim(0, 600)
ax3b.set_xlim(0, 600)

ax2.set_title("SIZ", fontsize=18)
ax3.set_title("SAZ", fontsize=18)
ax.axhline(y=137, color="black", linestyle="--", label="Winter MLD")
ax2.axhline(y=137, color="black", linestyle="--")
ax1.axhline(y=221, color="black", linestyle="--")
ax3.axhline(y=221, color="black", linestyle="--")

axb.set_xlabel("POC loss (mmol C m$^{-3}$ day$^{-1}$)", fontsize=15)
ax1b.set_xlabel("POC loss (mmol C m$^{-3}$ day$^{-1}$)", fontsize=15)#, fontweight="demibold")
ax2b.set_xlabel("POC flux (mg C m$^{-2}$ day$^{-1}$)", fontsize=15)
ax3b.set_xlabel("POC flux (mg C m$^{-2}$ day$^{-1}$)", fontsize=15)

ax.set_xlabel("Respiration rate (mmol C m$^{-3}$ day$^{-1}$)", fontsize=15)
ax1.set_xlabel("Respiration rate (mmol C m$^{-3}$ day$^{-1}$)", fontsize=15)
ax2.set_xlabel("")
ax3.set_xlabel("")

for a in [ax, ax1, ax2, ax3]:
    a.spines['bottom'].set_color(CB_color_cycle[4])
    a.spines['top'].set_visible(False)
    a.tick_params(axis='x', labelsize=11, colors=CB_color_cycle[4])
    a.xaxis.label.set_color(CB_color_cycle[4])
    a.set_ylabel("Depth", fontsize=14)
    a.set_yticks([20, 100, 200, 300, 400, 500, 600])
    a.tick_params(axis='y', labelsize=11)
    a.invert_yaxis()

for a in [axb, ax1b]:
    a.spines['top'].set_color(CB_color_cycle[0])
    a.spines['bottom'].set_visible(False)
    a.xaxis.tick_top()
    a.xaxis.set_label_position('top')
    a.tick_params(axis='x', labelsize=11, colors=CB_color_cycle[0])
    a.xaxis.label.set_color(CB_color_cycle[0])

for a in [ax2b, ax3b]:
    a.spines['top'].set_color('navy')
    a.spines['bottom'].set_visible(False)
    a.xaxis.tick_top()
    a.xaxis.set_label_position('top')
    a.tick_params(axis='x', labelsize=11, colors='navy')
    a.xaxis.label.set_color('navy')


# Add labels (a) and (b)
ax2.text(-0.1, 1.1, 'A)', transform=ax2.transAxes, fontsize=15, va='top', ha='right')
ax3.text(-0.1, 1.1, 'B)', transform=ax3.transAxes, fontsize=15, va='top', ha='right')
ax.text(-0.1, 1.1, 'C)', transform=ax.transAxes, fontsize=15, va='top', ha='right')
ax1.text(-0.1, 1.1, 'D)', transform=ax1.transAxes, fontsize=15, va='top', ha='right')


fig.legend( bbox_to_anchor=[0.45, 0.7], 
           loc='upper right')


fig.tight_layout()
ax.grid()
ax1.grid()


## Figure 6

In [ ]:
plt.rcParams.update({'font.size': 13})
fig, ((ax, ax4), (ax1, ax5), (ax2, ax3))=plt.subplots(3,2, figsize=(10,12))
var_y="bNCP_ML"
var_x="LiC_sum_cafe"
ds=ds_bNCP_EP.where(ds_bNCP_EP[var_x]>0, drop=True)
ds[var_x+"_molC"]=ds[var_x]/1000/12*30.4 #convert to get units of mol C m-2 yr-1 (30.4 is mean days in a month)
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")# _need_intercept=True)
y_pred = results["predict"]
r=results["r"]#[0]
ax.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax, x=var_x+"_molC", y=var_y, hue="CAFE_count", cmap="cividis",s=35, cbar_kwargs={"label": "# of months with valid EP"})
ax.set_ylabel("bNCP$_{ML}$ (mol C m$^{-2}$ szn$^{-1}$)", fontsize=14)
ax.set_xlabel("Li and Cassar EP mol C m$^{-2}$ yr$^{-1}$", fontsize=14)
ax.set_ylim(0,7)
ax.set_xlim(0,7)

ax.plot([0,7], [0,7], color="gray", linestyle="dashed")

del results, y_pred
var_x="Laws3_sum_cafe"
ds=ds_bNCP_EP.where(ds_bNCP_EP[var_x]>0, drop=True)

ds[var_x+"_molC"]=ds[var_x]/1000/12*30.4
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")# _need_intercept=True)
y_pred = results["predict"]
r=results["r"]#[0]
ax1.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax1, x=var_x+"_molC", y=var_y, hue="CAFE_count", cmap="cividis" ,s=35, cbar_kwargs={"label": "# of months with valid EP"})
ax1.set_ylabel("bNCP$_{ML}$ (mol C m$^{-2}$ szn$^{-1}$)", fontsize=14)
ax1.set_xlabel("Laws EP mol C m$^{-2}$ yr$^{-1}$", fontsize=14)
ax1.set_ylim(0,7)
ax1.set_xlim(0,7)
ax1.plot([0,7], [0,7], color="gray", linestyle="dashed")
ax1.set_title("")

### now BRIT EP
del results, y_pred
var_x="Brit_sum_cafe"
ds=ds_bNCP_withEP.where(ds_bNCP_withEP[var_x]>0, drop=True)
ds[var_x+"_molC"]=ds[var_x]/1000*30.4 # note, units are already molar
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")
y_pred = results["predict"]
r=results["r"]
ax2.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax2, x=var_x+"_molC", y=var_y, hue="CAFE_count", cmap="cividis" ,s=35, cbar_kwargs={"label": "# of months with valid EP"})
ax2.set_ylabel("bNCP$_{ML}$ (mol C m$^{-2}$ szn$^{-1}$)", fontsize=14)
ax2.set_xlabel("Brit EP mol C m$^{-2}$ yr$^{-1}$", fontsize=14)
ax2.set_ylim(0,7)
ax2.set_xlim(0,7)
ax2.plot([0,7], [0,7], color="gray", linestyle="dashed")
ax2.set_title("")

del results, y_pred
#########now ANCP
var_x="LiC_sum_cafe"
var_y="ANCP_600"
ds_dropna=ds_ANCP_EP.where(ds_ANCP_EP.ANCP_600.isnull()==False, drop=True)
ds=ds_dropna.where(ds_dropna[var_x]>0, drop=True)

ds[var_x+"_molC"]=ds[var_x]/1000/12*30.4
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")# _need_intercept=True)
y_pred = results["predict"]
r=results["r"]#[0]
ax4.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax4, x=var_x+"_molC", y=var_y, hue="fResp_600", alpha=1, s=35, cbar_kwargs={"label": "fraction respired and re-entrained"})
ax4.set_ylabel("ANCP (mol C m$^{-2}$ yr$^{-1}$)", fontsize=14)
ax4.set_xlabel("Li and Cassar EP mol C m$^{-2}$ yr$^{-1}$", fontsize=14)
ax4.set_title("")
ax4.plot([0,7], [0,7], color="gray", linestyle="dashed")
ax4.set_ylim(0,7)
ax4.set_xlim(0,7)

del results, y_pred
var_x="Laws3_sum_cafe"
ds[var_x+"_molC"]=ds[var_x]/1000/12*30.4
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")# _need_intercept=True)
y_pred = results["predict"]
r=results["r"]#[0]
ax5.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax5, x=var_x+"_molC", y=var_y, hue="fResp_600", alpha=1, s=35, cbar_kwargs={"label": "fraction respired and re-entrained"})
ax5.set_ylabel("ANCP (mol C m$^{-2}$ yr$^{-1}$)", fontsize=14)
ax5.set_xlabel("Laws EP molC m$^{-2}$ yr$^{-1}$", fontsize=14)
ax5.set_title("")
ax5.plot([0,7], [0,7], color="gray", linestyle="dashed")
ax5.set_ylim(0,7)
ax5.set_xlim(0,7)

## now BRIT EP 
del results, y_pred
var_x="Brit_sum_cafe"
ds[var_x+"_molC"]=ds[var_x]/1000*30.4
results = regress2(ds[var_x+"_molC"].values, ds[var_y].values, _method_type_2="major axis")
y_pred = results["predict"]
r=results["r"]#[0]
ax3.plot(ds[var_x+"_molC"], y_pred, color ='k', label="r$^2$="+str(np.round(r**2,2)))
ds.plot.scatter(ax=ax3, x=var_x+"_molC", y=var_y, hue="fResp_600", alpha=1, s=35, cbar_kwargs={"label": "fraction respired and re-entrained"})
ax3.set_ylabel("ANCP (mol C m$^{-2}$ yr$^{-1}$)", fontsize=14)
ax3.set_xlabel("Brit EP mol C m$^{-2}$ yr$^{-1}$", fontsize=14)
ax3.set_title("")
ax3.plot([0,7], [0,7], color="gray", linestyle="dashed")
ax3.set_ylim(0,7)
ax3.set_xlim(0,7)

ax.set_title("Annual EP vs bNCP$_{ML}$", fontsize=16)
ax4.set_title("Anuual EP vs ANCP", fontsize=16)

ax.text(-0.1, 1.1, 'A)', transform=ax.transAxes, fontsize=15, va='top', ha='right')
ax4.text(-0.1, 1.1, 'B)', transform=ax4.transAxes, fontsize=15, va='top', ha='right')
ax1.text(-0.1, 1.1, 'C)', transform=ax1.transAxes, fontsize=15, va='top', ha='right')
ax5.text(-0.1, 1.1, 'D)', transform=ax5.transAxes, fontsize=15, va='top', ha='right')
ax2.text(-0.1, 1.1, 'E)', transform=ax2.transAxes, fontsize=15, va='top', ha='right')
ax3.text(-0.1, 1.1, 'F)', transform=ax3.transAxes, fontsize=15, va='top', ha='right')

ax.legend()
ax1.legend()
ax4.legend()
ax5.legend()
ax2.legend()
ax3.legend()
fig.tight_layout()



## Supplemental

In [ ]:
## ammended to include subplot with mld last max for each zone
fig, (ax, ax1) = plt.subplots(1,2, figsize=(9,4))
# List of zones to include
zones_to_include = [ "SIZ","AZ", "SAZ", "STZ"]
combined_df=ds_ANCP_EP.to_dataframe()[['zone', 'fResp_600']]
filtered_df = combined_df[combined_df['zone'].isin(zones_to_include)]
# Create a box plot
color_palette = sns.color_palette("colorblind")
sns.boxplot(ax=ax, x='zone', y='fResp_600', color="darkgreen", data=filtered_df.reset_index(), palette=color_palette, order=["SIZ","AZ", "SAZ", "STZ"])#, boxprops=dict(alpha=.8))
ax.set_xlabel('Zone', fontsize=14)
ax.set_ylabel('fRRC', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=14)

combined_df=ds_ANCP_EP.to_dataframe()[['zone', 'mld_lastmax']]
filtered_df = combined_df[combined_df['zone'].isin(zones_to_include)]
sns.boxplot(ax=ax1, x='zone', y='mld_lastmax', color="darkgreen", data=filtered_df.reset_index(), palette=color_palette, order=["SIZ","AZ", "SAZ", "STZ"])#, boxprops=dict(alpha=.8))
ax1.set_xlabel('Zone', fontsize=14)
ax1.set_ylabel('Winter MLD (m)', fontsize=14)
ax1.tick_params(axis='both', which='major', labelsize=14)

fig.tight_layout()



In [ ]:

fig, (ax, ax1)=plt.subplots(1,2, figsize=(8,4))
sns.histplot((ds_ANCP_EP.ANCP_600+ds_ANCP_EP.RRC).to_series(), ax=ax, color="purple", alpha=0.6, label="ANCP+C respired \n and re-entrained")
sns.histplot(ds_bNCP_EP.bNCP_ML.to_series(), ax=ax, color="orange", alpha=0.6, label="bNCP$_{ML}$")
sns.histplot((ds_ANCP_EP.ANCP_600+ds_ANCP_EP.RRC).to_series(), ax=ax1, stat="density", color="purple", alpha=0.6, label="ANCP+C respired \n and re-entrained")
sns.histplot(ds_bNCP_EP.bNCP_ML.to_series(), ax=ax1, stat="density", color="orange", alpha=0.6, label="bNCP$_{ML}$")
ax.set_xlabel("mol C m$^{-2}$")
ax1.set_xlabel("mol C m$^{-2}$")
ax.legend(loc="upper right")
ax1.legend(loc="upper right")
fig.tight_layout()
tp1=ax.text(-0.1, 1.1, 'A)', transform=ax.transAxes, fontsize=15, va='top', ha='right')

tp2=ax1.text(-0.1, 1.1, 'B)', transform=ax1.transAxes, fontsize=15, va='top', ha='right')



In [ ]:
## compare bNCP_ML to bNCP integrated to maximum MLD during float season - see https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2023GL103459 for bNCP integrated methods
## they are both calculated from the same nitrate timeseries so have same Jan lat
fig, ax = plt.subplots(figsize=(7,5))
im=ax.scatter(ax=ax, x=bNCP_integrated, y=ds_bNCP_EP.bNCP_ML, c=ds_bNCP_EP.Jan_lat)
ax.plot([0,11],[0,11])
ax.set_ylim(0,11)
ax.set_xlim(0,11)
ax.set_ylabel("bNCP$_{ML}$", fontsize=14)
ax.set_xlabel("bNCP$_{integrated}$", fontsize=14)
cbar=plt.colorbar(im)
cbar.set_label("Latitude", fontsize=14)
fig.savefig(file_path+"/FigS3_1224.pdf")



In [ ]:
merged_ds["bNCP_ANCP_diff"]=merged_ds.bNCP_ML-merged_ds.ANCP_600
merged_ds["Resp_total"]=merged_ds.RRC+merged_ds.ANCP_600

#bin merged_ds by bNCP mol C, plot tot Resp and ANCP 600 vs bNCP ML
molC_bins=np.quantile(merged_ds.bNCP_ML.values, [0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1])
fRresp_bins=np.quantile(merged_ds.fResp_600.values, [0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1])
fig, (ax, axb) = plt.subplots(1,2, figsize=(12,6))
merged_ds.groupby_bins("bNCP_ML", molC_bins).mean().plot.scatter(ax=ax, x="bNCP_ML", y="Resp_total", s=55, color="navy")
merged_ds.plot.scatter(ax=ax, x="bNCP_ML", y="Resp_total", color="navy", s=30, alpha=0.25)

# Plot 1:1 line 
ax.plot([0, 12], [0, 12], label="1:1 line", color="gray", linestyle="dashed")
#linear regresssion 
regr = LinearRegression()
x=merged_625.groupby_bins("bNCP_ML", molC_bins).mean().bNCP_ML
y=merged_625.groupby_bins("bNCP_ML", molC_bins).mean().NCPSum_total_600_newDif
regr.fit(x.values.reshape(-1, 1), y.values)
y_pred = regr.predict(x.values.reshape(-1, 1))
ax.plot(x.values, y_pred, color="navy")
print(regr.score(x.values.reshape(-1, 1), y.values))


ax.set_ylim(0,12)
ax.set_xlim(0,12)


ax.legend(fontsize=15, loc="lower right")
ax.set_title("Float seasons with both bNCP$_{ML}$ and ANCP", fontsize=15)
ax.set_xlabel("bNCP$_{ML}$", fontsize=15)
ax.set_ylabel("RRC + ANCP", color="navy", fontsize=15)
ax.spines['left'].set_color('navy')
ax.spines['left'].set_linewidth(2)
ax.tick_params(axis='y', colors='navy', labelsize=14)
ax.tick_params(axis='x', labelsize=14)

merged_ds.groupby_bins("fResp_600", fRresp_bins).mean().plot.scatter(ax=axb, x="bNCP_ANCP_diff", y="fResp_600", s=55, color="cornflowerblue")
merged_ds.plot.scatter(ax=axb, x="bNCP_ANCP_diff", y="fResp_600", s=30, color="cornflowerblue", alpha=0.25)

axb.set_ylabel("fRRC", fontsize=15)
axb.set_xlabel("bNCP$_{ML}$ - ANCP (mol C m$^2$)", fontsize=15)

axb.tick_params(axis='y',  labelsize=14)
axb.tick_params(axis='x',  labelsize=14)

del regr
regr = LinearRegression()
x=merged_625.groupby_bins("fResp_600", fRresp_bins).mean().bNCP_ANCP_diff_newDif
y=merged_625.groupby_bins("fResp_600", fRresp_bins).mean().fResp_600
regr.fit(x.values.reshape(-1, 1), y.values)
y_pred = regr.predict(x.values.reshape(-1, 1))
axb.plot(x.values, y_pred, color="cornflowerblue")
print(regr.score(x.values.reshape(-1, 1), y.values))
axb.legend(fontsize=15)

fig.tight_layout()



In [ ]:
# to get # of months with over a third of float seasons with respiration
# ex with SIZ, repeat for Saz, then plot 
Jan_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Jan.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Feb_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Feb.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Mar_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Mar.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Apr_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Apr.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
May_saz_percent=DF_SAZ_zbinTWR.twr_molm3_May.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Jun_saz_percent=DF_SAZ_zbinTWR.twr_molm3_June.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Jul_saz_percent=DF_SAZ_zbinTWR.twr_molm3_July.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Aug_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Aug.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Sep_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Sept.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Oct_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Oct.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Nov_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Nov.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15
Dec_saz_percent=DF_SAZ_zbinTWR.twr_molm3_Dec.groupby("depth_center").count().to_xarray().sel(depth_center=slice(0,600))/15

print(Jan_saz_percent.where(Jan_saz_percent>0.33, other=0).values[0:5], "Jan")
print(Feb_saz_percent.where(Feb_saz_percent>0.33, other=0).values[0:5], "Feb")
print(Mar_saz_percent.where(Mar_saz_percent>0.33, other=0).values[0:5], "mar")
print(Apr_saz_percent.where(Apr_saz_percent>0.33, other=0).values[0:5], "Apr")
print(May_saz_percent.where(May_saz_percent>0.33, other=0).values[0:5], "may")
print(Jun_saz_percent.where(Jun_saz_percent>0.33, other=0).values[0:5], "Jun")
print(Jul_saz_percent.where(Jul_saz_percent>0.33, other=0).values[0:5], "July")
print(Aug_saz_percent.where(Aug_saz_percent>0.33, other=0).values[0:5], "Aug")
print(Sep_saz_percent.where(Sep_saz_percent>0.33, other=0).values[0:5], "Sep")
print(Oct_saz_percent.where(Oct_saz_percent>0.33, other=0).values[0:5], "oct")
print(Nov_saz_percent.where(Nov_saz_percent>0.33, other=0).values[0:5], "Nov")
print(Dec_saz_percent.where(Dec_saz_percent>0.33, other=0).values[0:5], "dec")



In [ ]:
#read in EKE files - see methods for datasets
#All_SO_EKE_13_22= 
#Auger_Geo_Anom_South= path

EKE_13_22_mean_passed=[]
for flt_szn in range(len(ds_ANCP_EP.seasonID.values)):
    lat_in=ds_ANCP_EP.isel(seasonID=flt_szn).Lat_jan.values
    lon_180_in=ds_ANCP_EP.isel(seasonID=flt_szn).Lon_180_jan.values
    if lat_in>-60:
        EKE_13_22_mean_passed.append(All_SO_EKE_13_22.sel(latitude=lat_in, longitude=lon_180_in, method="nearest").mean().values)
    elif lat_in<=-60: 
        lon_diff=abs(Auger_Geo_Anom_South.longitude-lon_180_in)
        lat_diff=abs(Auger_Geo_Anom_South.latitude-lat_in)
        dist_weight = (lon_diff**2 + lat_diff**2)**0.5
        EKE_match=Auger_Geo_Anom_South.where(dist_weight==dist_weight.min(), drop=True)
        EKE_13_22_mean_passed.append(EKE_match.EKE.mean().values)

## repeat for all the float years that had full annual cycles INCLUDING floats that did not pass criteria 
## remove any inf points
## concat pre and post
s_EKE_13_22_mean_passed = pd.series(EKE_13_22_mean_passed)
s_EKE_13_22_mean_all = pd.series(EKE_13_22_mean_all)

df_EKEplot = pd.concat([s_EKE_13_22_mean_all, s_EKE_13_22_mean_passed], axis=1)

fig, ax =plt.subplots()

ax = sns.barplot(df_EKEplot.rename({"EKE_13_22_mean":"post-criteria", "0":"pre-criteria"}), ax=ax)
sns.set(font_scale=2)
ax.set_ylabel("EKE cm$^{2}$ s$^{-2}$")

## for map, read in bsose data (iter 156) - plot "all" and "passed criteria" on map (see map code above)
